# Solar Filament Segmentation Challenge 2026
## U-Net + ConvNeXt training, evaluation, and inference

This notebook is the pipeline driver: it installs the extra deep-learning
dependencies, points the shared config at Kaggle's `/kaggle/input` mounts,
then calls the same `scripts/train.py` / `scripts/evaluate.py` / `scripts/infer.py`
used locally -- so the logic lives in one place (the code dataset attached to
this kernel), not duplicated into notebook cells.

**Task**: binary, category-agnostic filament segmentation (the competition's
Overview page states class labels are out of scope for scoring). A U-Net with a
ConvNeXt-Tiny encoder predicts a foreground probability map; watershed
post-processing splits it into individual filament instances for the
Panoptic-Quality-scored submission format.

### 0. Confirm the mounted data layout
Adjust `configs/config_kaggle.yaml`'s `data:` paths if this doesn't match.

In [ ]:
!find /kaggle/input -maxdepth 3

### 1. Bring in the code dataset and install missing dependencies

In [ ]:
!mkdir -p /kaggle/working/run
!cp -r /kaggle/input/filament-segmentation-code/* /kaggle/working/run/
%cd /kaggle/working/run
!ls

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

### 2. Train
`configs/config_kaggle.yaml` sets `image_size: 2048` (native resolution) and
`batch_size: 8` (auto-splits across both GPUs via `nn.DataParallel` when Kaggle
gives you a T4x2 accelerator, ~4 images/GPU; falls back to a single GPU
automatically if only one is visible). Override any config value from the CLI,
e.g. `--epochs` / `--batch-size`, without editing the file.

In [ ]:
!python scripts/train.py --config configs/config_kaggle.yaml

### 3. Training curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('outputs/logs/train_log.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log['epoch'], log['train_loss'], label='train_loss')
axes[0].plot(log['epoch'], log['val_loss'], label='val_loss')
axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].set_title('Loss')
axes[1].plot(log['epoch'], log['val_iou'], label='val_iou')
axes[1].plot(log['epoch'], log['val_dice_torchmetrics'], label='val_dice')
axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].set_title('Val metrics')
fig.tight_layout()
plt.show()

### 4. Evaluate (instance-level Panoptic Quality + Dice)
Runs the full inference + post-processing pipeline against the held-out val split's
ground truth -- the same instance-splitting logic used for the actual submission --
as a local sanity check against the competition's real leaderboard metric.

In [ ]:
!python scripts/evaluate.py --checkpoint outputs/checkpoints/best.pt

### 5. Inference -> submission CSV
Predicts on the test set, splits into filament instances, RLE-encodes each one, and
writes `outputs/submissions/submission_<timestamp>.csv` in the exact
`filament_id,segmentation_rle` format the competition expects.

In [ ]:
!python scripts/infer.py --checkpoint outputs/checkpoints/best.pt --visualize 10

In [ ]:
import glob
import pandas as pd

submissions = sorted(glob.glob('outputs/submissions/*.csv'))
print('Latest submission:', submissions[-1] if submissions else 'none found')
pd.read_csv(submissions[-1]).head()

### 6. Spot-check a few predictions

In [ ]:
import glob
from PIL import Image

for path in sorted(glob.glob('outputs/eda/predictions/*.png'))[:3]:
    display(Image.open(path))

### Outputs
- `outputs/checkpoints/best.pt` / `last.pt` -- model weights
- `outputs/logs/train_log.csv` -- per-epoch metrics
- `outputs/submissions/submission_*.csv` -- ready to submit to the competition
- `outputs/eda/predictions/*.png` -- qualitative overlays

All of the above persist as this notebook's **Output** after a committed run,
downloadable via `kaggle kernels output <username>/filament-unet-convnext-training -p <local-dir>`.